# 📦 Batch 1 & 5 - 2일 작업용 노트북

## ⚙️ 설정:
- **오늘 작업**: Batch 1 (약 357개 종목)
- **내일 작업**: Batch 5 (약 357개 종목)
- **API 키**: 2개 (API 키 1-2번) - 양일 동일 키 사용
- **예상 시간**: 각 12.5시간

## 📋 사용 방법:

### ☀️ 오늘:
1. **Cell 1-8**: 순서대로 실행 (함수 정의)
2. **Cell 10**: API 키 입력 + 오늘 작업 실행

### 🌙 내일:
1. **Cell 1-8**: 이미 실행됨 (재실행 필요 없음)
2. **Cell 12**: API 키 입력 (동일 키) + 내일 작업 실행

---

## 📚 Step 1: 라이브러리 Import

In [20]:
import os
import time
import json
import zipfile
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from typing import Dict, List
import FinanceDataReader as fdr
from datetime import datetime
import numpy as np
import warnings

warnings.filterwarnings('ignore', category=pd.errors.SettingWithCopyWarning)

# DART API URL
DART_CORP_CODE_URL = "https://opendart.fss.or.kr/api/corpCode.xml"
FNLTT_URL = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
STOCK_TOTQY_URL = "https://opendart.fss.or.kr/api/stockTotqySttus.json"

# 보고서 코드
REPRT_MAP = {"11013": "1Q", "11012": "2Q(반기)", "11014": "3Q", "11011": "Annual"}
ANNUAL_CODE = ["11011"]
QUARTERLY_CODES = ["11013", "11012", "11014"]
NUM_COLS_CANDIDATES = ["thstrm_amount", "frmtrm_amount", "bfefrmtrm_amount", "thstrm_add_amount", "frmtrm_add_amount", "bfefrmtrm_add_amount"]
END_YEAR = datetime.now().year
SAVE_DIR = "./data"
DAILY_LIMIT = 39500

print("✅ Import 완료")

✅ Import 완료


## 🔄 Step 2: API 키 관리자 & 진행 상황 관리자

In [21]:
class APIKeyManager:
    def __init__(self, api_keys):
        self.api_keys = [k for k in api_keys if k and "여기에" not in k]
        if not self.api_keys:
            raise ValueError("❌ API 키를 입력해주세요!")
        self.current_index = 0
        self.usage = {key: 0 for key in self.api_keys}
        self.daily_limit = DAILY_LIMIT
        print(f"✅ API 키 {len(self.api_keys)}개 로드 완료")
    
    def get_key(self):
        current_key = self.api_keys[self.current_index]
        if self.usage[current_key] >= self.daily_limit:
            old_index = self.current_index
            self.switch_next()
            print(f"🔄 API 키 전환: Key #{old_index+1} → Key #{self.current_index+1}")
        return self.api_keys[self.current_index]
    
    def record_call(self):
        self.usage[self.api_keys[self.current_index]] += 1
    
    def switch_next(self):
        self.current_index = (self.current_index + 1) % len(self.api_keys)
        if all(u >= self.daily_limit for u in self.usage.values()):
            raise RuntimeError("⚠️ 모든 API 키 한도 도달!")
    
    def get_stats(self):
        return {"current_key_index": self.current_index + 1, "usage": {f"Key_{i+1}": u for i, (k, u) in enumerate(self.usage.items())}}

class ProgressManager:
    def __init__(self, progress_file):
        self.progress_file = progress_file
        self.completed = set()
        self.failed = {}
        self.load()
    
    def load(self):
        if os.path.exists(self.progress_file):
            with open(self.progress_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                self.completed = set(data.get('completed', []))
                self.failed = data.get('failed', {})
            print(f"📂 진행 로드: 완료 {len(self.completed)}개")
    
    def save(self):
        with open(self.progress_file, 'w', encoding='utf-8') as f:
            json.dump({'completed': list(self.completed), 'failed': self.failed, 'last_updated': datetime.now().isoformat()}, f, ensure_ascii=False, indent=2)
    
    def mark_completed(self, code):
        self.completed.add(code)
        if code in self.failed:
            del self.failed[code]
    
    def mark_failed(self, code, error):
        self.failed[code] = str(error)
    
    def is_completed(self, code):
        return code in self.completed

print("✅ 관리자 정의 완료")

✅ 관리자 정의 완료


## 🛠️ Step 3: 유틸리티 함수

In [22]:
def choose_excel_engine():
    try:
        import xlsxwriter
        return "xlsxwriter"
    except ImportError:
        return "openpyxl"

def request_with_retry(method, url, params=None, max_retries=5, timeout=20):
    attempt = 0
    while True:
        try:
            resp = requests.request(method=method, url=url, params=params, timeout=timeout)
            resp.raise_for_status()
            return resp
        except requests.RequestException as e:
            attempt += 1
            if attempt > max_retries:
                raise RuntimeError(f"HTTP 요청 실패: {e}") from e
            time.sleep(min(2 ** attempt * 0.5, 10.0))

def ensure_corp_code_xml(api_key):
    xml_path = "CORPCODE.xml"
    if not os.path.exists(xml_path):
        print("기업 고유 코드 파일 다운로드 중...")
        resp = request_with_retry("GET", DART_CORP_CODE_URL, params={"crtfc_key": api_key})
        with open("corp_code.zip", "wb") as f:
            f.write(resp.content)
        with zipfile.ZipFile("corp_code.zip", "r") as zf:
            zf.extractall("./")
    return xml_path

def load_corp_codes(api_key):
    xml_file = ensure_corp_code_xml(api_key)
    tree = ET.parse(xml_file)
    root = tree.getroot()
    corp_dict = {}
    for el in root.iter("list"):
        corp_name = el.findtext("corp_name")
        corp_code = el.findtext("corp_code")
        stock_code = el.findtext("stock_code")
        if stock_code:
            corp_dict[stock_code] = {"corp_name": corp_name, "corp_code": corp_code}
    return corp_dict

def reorder_columns(df):
    if df is None or df.empty:
        return df
    preferred_front = ["account_id", "account_nm", "sj_div", "sj_nm", "bsns_year", "reprt_code", "reprt_name", "fs_div", "thstrm_amount", "frmtrm_amount", "bfefrmtrm_amount", "corp_name", "corp_code"]
    cols_exist = [c for c in preferred_front if c in df.columns]
    cols_rest = [c for c in df.columns if c not in cols_exist]
    return df[cols_exist + cols_rest]

def get_listing_year(stock_code):
    try:
        df_krx = fdr.StockListing('KRX')
        stock_info = df_krx[df_krx['Code'] == stock_code]
        if not stock_info.empty and 'ListingDate' in stock_info.columns:
            return pd.to_datetime(stock_info.iloc[0]['ListingDate']).year
        return 2000
    except:
        return 2000

print("✅ 유틸리티 함수 정의 완료")

✅ 유틸리티 함수 정의 완료


## 📊 Step 4: DART 재무제표 크롤링 함수

In [23]:
def fetch_fnltt(corp_code, year, reprt_code, fs_div, api_key):
    params = {"crtfc_key": api_key, "corp_code": corp_code, "bsns_year": year, "reprt_code": reprt_code, "fs_div": fs_div}
    for attempt in range(6):
        resp = request_with_retry("GET", FNLTT_URL, params=params)
        try:
            data = resp.json()
        except ValueError:
            if attempt >= 5:
                return {"status": "999", "message": "JSON error", "list": []}
            time.sleep(1.0 + attempt * 0.5)
            continue
        status = data.get("status")
        if status in {"000", "013"}:
            return data
        if attempt >= 5:
            return data
        time.sleep(min(2 ** attempt * 0.7, 10.0))
    return {"status": "999", "message": "unknown error", "list": []}

def fetch_range_by_fs(stock_code, start_year, end_year, fs_div, corp_dict, api_manager):
    if stock_code not in corp_dict:
        return pd.DataFrame()
    corp_code = corp_dict[stock_code]["corp_code"]
    corp_name = corp_dict[stock_code]["corp_name"]
    all_rows = []
    for year in range(start_year, end_year + 1):
        for reprt_code in (QUARTERLY_CODES + ANNUAL_CODE):
            data = fetch_fnltt(corp_code, year, reprt_code, fs_div, api_manager.get_key())
            api_manager.record_call()
            if data.get("status") != "000":
                continue
            for r in data.get("list") or []:
                r["reprt_name"] = REPRT_MAP.get(str(r.get("reprt_code")), str(r.get("reprt_code")))
                r["fs_div"] = fs_div
                r["corp_code"] = corp_code
                r["corp_name"] = corp_name
                all_rows.append(r)
            time.sleep(0.35)
    df = pd.DataFrame(all_rows)
    if df.empty:
        return df
    for col in NUM_COLS_CANDIDATES:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(",", "").str.replace(" ", "").replace({"": None, "-": None})
            df[col] = df[col].str.replace(r"^\((.*)\)$", r"-\1", regex=True)
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df.sort_values([c for c in ["bsns_year", "reprt_code"] if c in df.columns]).reset_index(drop=True)

print("✅ 재무제표 크롤링 함수 정의 완료")

✅ 재무제표 크롤링 함수 정의 완료


## 📈 Step 5: DART 발행주식수 크롤링 함수

In [24]:
def fetch_stock_totqy(corp_code, year, reprt_code, api_key):
    params = {"crtfc_key": api_key, "corp_code": corp_code, "bsns_year": year, "reprt_code": reprt_code}
    for attempt in range(6):
        resp = request_with_retry("GET", STOCK_TOTQY_URL, params=params)
        try:
            data = resp.json()
        except ValueError:
            if attempt >= 5:
                return {"status": "999", "list": []}
            time.sleep(1.0)
            continue
        if data.get("status") in {"000", "013"}:
            return data
        if attempt >= 5:
            return data
        time.sleep(min(2 ** attempt * 0.7, 10.0))
    return {"status": "999", "list": []}

def fetch_stock_totqy_range(stock_code, start_year, end_year, corp_dict, api_manager):
    if stock_code not in corp_dict:
        return pd.DataFrame()
    corp_code = corp_dict[stock_code]["corp_code"]
    all_rows = []
    for year in range(start_year, end_year + 1):
        for reprt_code in ANNUAL_CODE:
            data = fetch_stock_totqy(corp_code, year, reprt_code, api_manager.get_key())
            api_manager.record_call()
            if data.get("status") != "000":
                continue
            for r in data.get("list") or []:
                r["bsns_year"] = year
                all_rows.append(r)
            time.sleep(0.35)
    df = pd.DataFrame(all_rows)
    for col in ["istc_totqy", "tesstk_co", "distb_stock_co"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

print("✅ 발행주식수 크롤링 함수 정의 완료")

✅ 발행주식수 크롤링 함수 정의 완료


## 💹 Step 6: 주가 데이터 크롤링 함수

In [25]:
def fetch_stock_price(stock_code, start_date, end_date):
    try:
        df = fdr.DataReader(stock_code, start_date, end_date)
        if df.empty:
            return pd.DataFrame()
        df = df.reset_index()
        df['stock_code'] = stock_code
        return df
    except:
        return pd.DataFrame()

def calculate_market_cap_by_year(df_price, df_stock, start_year, end_year):
    if df_price.empty:
        return pd.DataFrame()
    market_cap_data = []
    for year in range(start_year, end_year + 1):
        try:
            df_price_copy = df_price.copy()
            df_price_copy['Date'] = pd.to_datetime(df_price_copy['Date'])
            year_data = df_price_copy[df_price_copy['Date'].dt.year == year].copy()
            if year_data.empty:
                continue
            target_date = pd.to_datetime(f"{year}-12-31")
            year_data['date_diff'] = abs((year_data['Date'] - target_date).dt.days)
            closest_date = year_data.loc[year_data['date_diff'].idxmin()]
            shares = None
            if not df_stock.empty and 'istc_totqy' in df_stock.columns:
                stock_year = df_stock[df_stock['bsns_year'] == year]
                if not stock_year.empty:
                    shares = stock_year.iloc[0]['istc_totqy']
            market_cap = closest_date['Close'] * shares if shares and not pd.isna(shares) else None
            market_cap_data.append({'year': year, 'date': closest_date['Date'].strftime('%Y-%m-%d'), 'close_price': closest_date['Close'], 'shares': shares, 'market_cap': market_cap, 'market_cap_billion': market_cap / 1e9 if market_cap else None})
        except:
            continue
    return pd.DataFrame(market_cap_data)

print("✅ 주가 크롤링 함수 정의 완료")

✅ 주가 크롤링 함수 정의 완료


## 💾 Step 7: Excel 저장 함수

In [26]:
def split_and_save_sheets(df, df_stock, df_price, stock_code, start_year, end_year, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    xlsx_path = os.path.join(save_dir, f"{stock_code}_재무제표_{start_year}_{end_year}.xlsx")
    df_market_cap = calculate_market_cap_by_year(df_price, df_stock, start_year, end_year)
    with pd.ExcelWriter(xlsx_path, engine=choose_excel_engine()) as writer:
        pd.DataFrame({"항목": ["파일 설명", "시트 구성"], "내용": ["DART 재무제표 + 주식 + 주가 + 시가총액", "CFS/OFS Annual/Quarterly, Stock_Info, Stock_Price, Market_Cap"]}).to_excel(writer, sheet_name="README", index=False)
        
        def _write(sheet_name, df_sub):
            if df_sub is not None and not df_sub.empty:
                reorder_columns(df_sub).to_excel(writer, sheet_name=sheet_name, index=False)
            else:
                pd.DataFrame({"메시지": ["데이터 없음"]}).to_excel(writer, sheet_name=sheet_name, index=False)
        
        cfs = df[df["fs_div"] == "CFS"] if "fs_div" in df.columns else pd.DataFrame()
        ofs = df[df["fs_div"] == "OFS"] if "fs_div" in df.columns else pd.DataFrame()
        _write("CFS_Annual", cfs[cfs["reprt_code"].isin(ANNUAL_CODE)] if not cfs.empty else pd.DataFrame())
        _write("CFS_Quarterly", cfs[cfs["reprt_code"].isin(QUARTERLY_CODES)] if not cfs.empty else pd.DataFrame())
        _write("OFS_Annual", ofs[ofs["reprt_code"].isin(ANNUAL_CODE)] if not ofs.empty else pd.DataFrame())
        _write("OFS_Quarterly", ofs[ofs["reprt_code"].isin(QUARTERLY_CODES)] if not ofs.empty else pd.DataFrame())
        _write("Stock_Info", df_stock)
        _write("Stock_Price", df_price)
        _write("Market_Cap", df_market_cap)

print("✅ Excel 저장 함수 정의 완료")

✅ Excel 저장 함수 정의 완료


---

# ☀️ 오늘 작업 - Batch 1

## 🔑 API 키 1-2번 입력 후 실행

**실행 전 확인:**
- Cell 1~7 모두 실행 완료
- batch_1_stocks.csv 파일 존재
- API 키 2개 준비

---

In [ ]:
# ==========================================
# ☀️ 오늘 작업: Batch 1
# ==========================================

# 🔑 API 키 입력 (2개)
API_KEYS = [
    "",
    "",
    
]

# 설정
BATCH_NUMBER = 1
BATCH_FILE = f"batch_{BATCH_NUMBER}_stocks.csv"
PROGRESS_FILE = f"progress_batch_{BATCH_NUMBER}.json"

print("="*70)
print("🚀 Batch 1 재무데이터 수집 시작 (☀️ 오늘 작업)")
print("="*70)

# 배치 파일 로드
if not os.path.exists(BATCH_FILE):
    raise FileNotFoundError(f"❌ {BATCH_FILE} 파일이 없습니다!")

df_batch = pd.read_csv(BATCH_FILE)
stock_codes = df_batch['Code'].astype(str).str.zfill(6).tolist()

print(f"📌 총 종목 수: {len(stock_codes)}개")
print(f"📁 저장 위치: {SAVE_DIR}")
print("="*70)

# 초기화
api_manager = APIKeyManager(API_KEYS)
progress_manager = ProgressManager(PROGRESS_FILE)
corp_dict = load_corp_codes(api_manager.get_key())

# 통계
total_stocks = len(stock_codes)
success_count = 0
skip_count = 0
fail_count = 0
start_time = datetime.now()

# 크롤링 시작
for idx, stock_code in enumerate(stock_codes, 1):
    if progress_manager.is_completed(stock_code):
        skip_count += 1
        continue
    
    stock_name = corp_dict.get(stock_code, {}).get("corp_name", "알 수 없음")
    print(f"\n[{idx}/{total_stocks}] {stock_code} ({stock_name})")
    
    try:
        listing_year = get_listing_year(stock_code)
        df_cfs = fetch_range_by_fs(stock_code, listing_year, END_YEAR, "CFS", corp_dict, api_manager)
        df_ofs = fetch_range_by_fs(stock_code, listing_year, END_YEAR, "OFS", corp_dict, api_manager)
        df_stock = fetch_stock_totqy_range(stock_code, listing_year, END_YEAR, corp_dict, api_manager)
        df_price = fetch_stock_price(stock_code, f"{listing_year}-01-01", f"{END_YEAR}-12-31")
        
        if df_cfs.empty and df_ofs.empty:
            print(f"  ⚠️ 데이터 없음")
            progress_manager.mark_completed(stock_code)
            skip_count += 1
            continue
        
        df_all = pd.concat([df_cfs, df_ofs], ignore_index=True)
        split_and_save_sheets(df_all, df_stock, df_price, stock_code, listing_year, END_YEAR, SAVE_DIR)
        progress_manager.mark_completed(stock_code)
        success_count += 1
        
        if idx % 10 == 0:
            progress_manager.save()
        
        elapsed = (datetime.now() - start_time).total_seconds()
        remaining = (elapsed / idx) * (total_stocks - idx) / 3600
        print(f"  ✅ 완료 | 진행: {idx}/{total_stocks} ({idx/total_stocks*100:.1f}%) | 남은시간: {remaining:.1f}h")
        
    except Exception as e:
        fail_count += 1
        progress_manager.mark_failed(stock_code, str(e))
        print(f"  ❌ 실패: {e}")

progress_manager.save()

print("\n" + "="*70)
print("✅ Batch 1 작업 완료!")
print(f"성공: {success_count} | 건너뜀: {skip_count} | 실패: {fail_count}")
print(f"소요시간: {(datetime.now() - start_time).total_seconds()/3600:.2f}시간")
print("="*70)

🚀 Batch 1 재무데이터 수집 시작 (☀️ 오늘 작업)
📌 총 종목 수: 358개
📁 저장 위치: ./data
✅ API 키 2개 로드 완료
📂 진행 로드: 완료 358개

✅ Batch 1 작업 완료!
성공: 0 | 건너뜀: 358 | 실패: 0
소요시간: 0.00시간


---

# 🌙 내일 작업 - Batch 5

## 🔑 동일 API 키 1-2번 입력 후 실행

**실행 전 확인:**
- Cell 1~7은 재실행 불필요 (이미 정의됨)
- batch_5_stocks.csv 파일 존재
- 동일 API 키 사용 (일일 한도 리셋됨)

---

In [ ]:
# ==========================================
# 🌙 내일 작업: Batch 5
# ==========================================

# 🔑 API 키 입력 (동일 키 재사용)
API_KEYS = [
    "",
    "",
    
]

# 설정
BATCH_NUMBER = 5
BATCH_FILE = f"batch_{BATCH_NUMBER}_stocks.csv"
PROGRESS_FILE = f"progress_batch_{BATCH_NUMBER}.json"

print("="*70)
print("🚀 Batch 5 재무데이터 수집 시작 (🌙 내일 작업)")
print("="*70)

# 배치 파일 로드
if not os.path.exists(BATCH_FILE):
    raise FileNotFoundError(f"❌ {BATCH_FILE} 파일이 없습니다!")

df_batch = pd.read_csv(BATCH_FILE)
stock_codes = df_batch['Code'].astype(str).str.zfill(6).tolist()

print(f"📌 총 종목 수: {len(stock_codes)}개")
print(f"📁 저장 위치: {SAVE_DIR}")
print("="*70)

# 초기화
api_manager = APIKeyManager(API_KEYS)
progress_manager = ProgressManager(PROGRESS_FILE)
corp_dict = load_corp_codes(api_manager.get_key())

# 통계
total_stocks = len(stock_codes)
success_count = 0
skip_count = 0
fail_count = 0
start_time = datetime.now()

# 크롤링 시작
for idx, stock_code in enumerate(stock_codes, 1):
    if progress_manager.is_completed(stock_code):
        skip_count += 1
        continue
    
    stock_name = corp_dict.get(stock_code, {}).get("corp_name", "알 수 없음")
    print(f"\n[{idx}/{total_stocks}] {stock_code} ({stock_name})")
    
    try:
        listing_year = get_listing_year(stock_code)
        df_cfs = fetch_range_by_fs(stock_code, listing_year, END_YEAR, "CFS", corp_dict, api_manager)
        df_ofs = fetch_range_by_fs(stock_code, listing_year, END_YEAR, "OFS", corp_dict, api_manager)
        df_stock = fetch_stock_totqy_range(stock_code, listing_year, END_YEAR, corp_dict, api_manager)
        df_price = fetch_stock_price(stock_code, f"{listing_year}-01-01", f"{END_YEAR}-12-31")
        
        if df_cfs.empty and df_ofs.empty:
            print(f"  ⚠️ 데이터 없음")
            progress_manager.mark_completed(stock_code)
            skip_count += 1
            continue
        
        df_all = pd.concat([df_cfs, df_ofs], ignore_index=True)
        split_and_save_sheets(df_all, df_stock, df_price, stock_code, listing_year, END_YEAR, SAVE_DIR)
        progress_manager.mark_completed(stock_code)
        success_count += 1
        
        if idx % 10 == 0:
            progress_manager.save()
        
        elapsed = (datetime.now() - start_time).total_seconds()
        remaining = (elapsed / idx) * (total_stocks - idx) / 3600
        print(f"  ✅ 완료 | 진행: {idx}/{total_stocks} ({idx/total_stocks*100:.1f}%) | 남은시간: {remaining:.1f}h")
        
    except Exception as e:
        fail_count += 1
        progress_manager.mark_failed(stock_code, str(e))
        print(f"  ❌ 실패: {e}")

progress_manager.save()

print("\n" + "="*70)
print("✅ Batch 5 작업 완료!")
print(f"성공: {success_count} | 건너뜀: {skip_count} | 실패: {fail_count}")
print(f"소요시간: {(datetime.now() - start_time).total_seconds()/3600:.2f}시간")
print("="*70)

🚀 Batch 5 재무데이터 수집 시작 (🌙 내일 작업)
📌 총 종목 수: 357개
📁 저장 위치: ./data
✅ API 키 2개 로드 완료
📂 진행 로드: 완료 357개

✅ Batch 5 작업 완료!
성공: 0 | 건너뜀: 357 | 실패: 0
소요시간: 0.00시간
